# Stage 2 — Supervised Instruction Fine-Tuning (SFT)
### Load the Stage-1 model from the Hub → teach it to follow instructions → push to the Hub

This is **notebook 2 of 3**. It downloads the model you pushed in Stage 1,
adds a fresh LoRA adapter, and fine-tunes on **instruction → response**
pairs, then merges and pushes the result.

**Concept — why "supervised"?** The training *engine* is the same next-token
prediction as Stage 1. What changes is the **data**: here every example is a
human-curated `(instruction, ideal answer)` pair — the answer **is the
label**. Learning from human-provided input→target pairs is the textbook
definition of *supervised* learning. So SFT is "supervised learning
implemented through the next-token interface".

```text
Stage 1 model (Hub)  →  + instruction data (labeled pairs)  →  push Stage 2 model (Hub)
```

In [1]:
# ============================================================
# Step 1. Install libraries
# ============================================================
!pip install -q -U datasets transformers accelerate peft bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 68.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.1 MB/s eta 0:00:00:00:0100:01


In [2]:
# ============================================================
# Step 2. Imports
# ============================================================
import os, gc, json
from dataclasses import dataclass, asdict

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)

## Hugging Face login

We push every stage's model to the Hub, and each later notebook **pulls the
previous stage's model from the Hub**. So you must be logged in.

Two ways to provide your token (create one at
https://huggingface.co/settings/tokens with *write* access):

- **Easiest in Colab:** open the key icon on the left, add a secret named
  `HF_TOKEN`, then run the cell below — it reads the secret automatically.
- **Or** just run `login()` and paste the token when prompted.

In [3]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Logged in as: Mohan143


## Step 3 — Configuration

Point `stage1_repo_name` at the repo you created in Stage 1. The SFT dataset
`lavita/ChatDoctor-HealthCareMagic-100k` already has `instruction` / `input`
/ `output` columns, so it drops straight into our Alpaca formatter.

In [4]:
# ============================================================
# Step 3. Configuration
# ============================================================
@dataclass
class Config:
    # Stage-1 output is THIS stage's base model.
    stage1_repo_name: str = "med-tinyllama-stage1-pretrained"

    # Instruction (SFT) dataset.
    dataset_name: str = "lavita/ChatDoctor-HealthCareMagic-100k"
    n_samples: int = 3000               # subsample for a fast Colab demo

    # Hub repo NAME for this stage's output.
    stage2_repo_name: str = "med-tinyllama-stage2-sft"
    private_repo: bool = True

    output_dir: str = "/content/stage2_output"
    adapter_dir: str = "/content/stage2_adapter"
    merged_dir: str = "/content/stage2_merged"

    max_length: int = 512
    test_size: float = 0.1
    seed: int = 42

    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-4         # a bit lower than Stage 1 (we are refining)
    warmup_steps: int = 5
    weight_decay: float = 0.01
    logging_steps: int = 5
    max_steps: int = -1


config = Config()
for d in (config.output_dir, config.adapter_dir, config.merged_dir):
    os.makedirs(d, exist_ok=True)

STAGE1_REPO = f"{HF_USERNAME}/{config.stage1_repo_name}"   # base for this stage
STAGE2_REPO = f"{HF_USERNAME}/{config.stage2_repo_name}"   # output of this stage
print("Base model (from Stage 1):", STAGE1_REPO)
print("Will push Stage 2 to:     ", STAGE2_REPO)

Base model (from Stage 1): Mohan143/med-tinyllama-stage1-pretrained
Will push Stage 2 to:      Mohan143/med-tinyllama-stage2-sft


## Step 4 — Load and format the instruction data (Alpaca template)

The model only ever sees **text**, so we render each record into a fixed
template. Using the **same template at train and inference time** is what
makes instruction tuning work.

```text
### Instruction:
{instruction}

### Response:
{output}
```

In [5]:
# ============================================================
# Step 4. Load instruction data, subsample, format to Alpaca text
# ============================================================
sft = load_dataset(config.dataset_name, split="train")
sft = sft.shuffle(seed=config.seed).select(range(min(config.n_samples, len(sft))))
print("Columns:", sft.column_names)


def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()
    if input_text:
        text = (f"### Instruction:\n{instruction}\n\n"
                f"### Input:\n{input_text}\n\n"
                f"### Response:\n{output_text}")
    else:
        text = (f"### Instruction:\n{instruction}\n\n"
                f"### Response:\n{output_text}")
    return {"text": text}


sft = sft.map(format_instruction_record, remove_columns=sft.column_names)
print("\nFormatted example:\n")
print(sft[0]["text"][:600])

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Columns: ['instruction', 'input', 'output']


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]


Formatted example:

### Instruction:
If you are a doctor, please answer the medical questions based on the patient's description.

### Input:
I have been having alot of catching ,pain and discomfort under my right rib.  If I twist to either side especially my right it feels like my rib actually catches on something and at times I have to stop try to catch my breath and wait for it to subside.  There are times if I am laughing too hard that it will do the same thing but normally its more so if I have twisted or moved  a certain way

### Response:
Hi thanks for asking question. Here you are complaining pain in part


In [6]:
# Train / validation split.
sft = sft.train_test_split(test_size=config.test_size, seed=config.seed)
sft["validation"] = sft.pop("test")
print(sft)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 300
    })
})


## Step 5 — Tokenize with padding and **label masking (`-100`)**

We pad each example to 512 tokens, then set every **padding** position in
the labels to `-100`. PyTorch ignores `-100` in the loss, so the model is
graded only on **real** tokens, never on padding.

In [7]:
# ============================================================
# Step 5. Tokenizer (from Stage-1 repo) + tokenize with -100 masking
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(STAGE1_REPO, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_instruction(examples):
    toks = tokenizer(examples["text"], truncation=True,
                     padding="max_length", max_length=config.max_length)
    toks["labels"] = [
        [t if m == 1 else -100 for t, m in zip(ids, attn)]
        for ids, attn in zip(toks["input_ids"], toks["attention_mask"])
    ]
    return toks


sft_tok = sft.map(tokenize_instruction, batched=True,
                  remove_columns=sft["train"].column_names,
                  desc="Tokenizing instruction data")
print(sft_tok)

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

Tokenizing instruction data:   0%|          | 0/2700 [00:00<?, ? examples/s]

Tokenizing instruction data:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300
    })
})


## Step 6 — Load the Stage-1 model from the Hub + a NEW LoRA adapter

We download the **merged Stage-1 model** (it already speaks medical) in
4-bit, and attach a *fresh* LoRA adapter that will learn instruction-following
on top of that domain knowledge.

In [8]:
# ============================================================
# Step 6. Stage-1 model (from Hub) + new LoRA adapter
# ============================================================
use_cuda = torch.cuda.is_available()
print("CUDA:", use_cuda)
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        STAGE1_REPO, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        STAGE1_REPO, torch_dtype=torch.float32, trust_remote_code=True)
base_model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=config.lora_r, lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

CUDA: True


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 7 — Train

In [ ]:
# ============================================================
# Step 7. Train the instruction adapter
# ============================================================
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps, logging_first_step=True,
    eval_strategy="steps", eval_steps=50,
    save_strategy="no",
    fp16=use_cuda, bf16=False, report_to="none", remove_unused_columns=False,
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=sft_tok["train"], eval_dataset=sft_tok["validation"],
                  data_collator=data_collator)
print("Training...")
trainer.train()
print("Done.")


Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,2.315335,2.267835
100,2.175119,2.224911
150,2.246544,2.193845
200,2.254564,2.174995
250,2.129919,2.157610
300,2.239017,2.146536
350,2.164533,2.138899
400,2.165798,2.130934
450,2.082398,2.125377
500,1.993556,2.122840


## Step 8 — Merge and push to the Hub

**Correctness point:** the adapter was trained on the Stage-1 model, so we
merge it back onto the **Stage-1 model** (not the raw TinyLlama) to keep the
domain knowledge intact. Stage 3 will load `STAGE2_REPO` as its base.

In [ ]:
# ============================================================
# Step 8. Save adapter, merge onto Stage-1 base, push to Hub
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

del trainer

In [ ]:

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

base_fp = AutoModelForCausalLM.from_pretrained(
    STAGE1_REPO,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None, trust_remote_code=True)
merged = PeftModel.from_pretrained(base_fp, config.adapter_dir).merge_and_unload()
merged.save_pretrained(config.merged_dir)
tokenizer.save_pretrained(config.merged_dir)

merged.push_to_hub(STAGE2_REPO, private=config.private_repo)
tokenizer.push_to_hub(STAGE2_REPO, private=config.private_repo)
print(f"Stage 2 (SFT) model pushed to: https://huggingface.co/{STAGE2_REPO}")

## Step 9 — Q&A inference

Now we prompt in the instruction template (ending with `### Response:`).

In [ ]:
# ============================================================
# Step 9. Instruction-style inference
# ============================================================
merged.eval()
device = merged.device


def ask(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction.strip()}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                              pad_token_id=tokenizer.eos_token_id,
                              eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)


for q in ["What is the primary mechanism of action of metformin?",
          "What are common side effects of statins?"]:
    print("=" * 90)
    print("Q:", q)
    print(ask(q))

## Done — on to Stage 3

Your instruction-following medical model is on the Hub at `STAGE2_REPO`.
Open **notebook 3 (DPO)** and set its `stage2_repo` to this id; it will align
the model to prefer better answers using chosen-vs-rejected pairs.